In [ ]:
from tree_sitter import Parser
from tree_sitter_language_pack import get_language


class PythonExtractor:

    def __init__(self):
        self.language = get_language("python")
        self.parser = Parser(self.language)

    def parse(self, source_code: bytes):
        return self.parser.parse(source_code)

    def extract(self, source_code: bytes):
        tree = self.parse(source_code)

        return {
            "type": "file",
            "start_line": 1,
            "end_line": source_code.count(b"\n") + 1,
            "children": self._extract_scope(
                tree.root_node,
                source_code
            )
        }

    def _extract_scope(self, node, source_code):

        entities = []

        for child in node.children:

            # -------------------------
            # Comments
            # -------------------------
            if child.type == "comment":
                entities.append({
                    "type": "comment",
                    "start_line": child.start_point[0] + 1,
                    "end_line": child.end_point[0] + 1,
                    "text": self._text(child, source_code)
                })

            # -------------------------
            # Class
            # -------------------------
            elif child.type == "class_definition":
                entities.append(
                    self._extract_class(child, source_code)
                )

            # -------------------------
            # Function
            # -------------------------
            elif child.type == "function_definition":
                entities.append(
                    self._extract_function(child, source_code)
                )

            # -------------------------
            # Decorated function/class
            # -------------------------
            elif child.type == "decorated_definition":
                entities.append(
                    self._extract_decorated_definition(
                        child,
                        source_code
                    )
                )

            # -------------------------
            # Conditional
            # -------------------------
            elif child.type == "if_statement":
                entities.append(
                    self._extract_control_flow(
                        child,
                        source_code,
                        "if"
                    )
                )

            # -------------------------
            # Loops
            # -------------------------
            elif child.type in (
                "for_statement",
                "while_statement"
            ):
                entities.append(
                    self._extract_control_flow(
                        child,
                        source_code,
                        child.type.replace("_statement", "")
                    )
                )

            # -------------------------
            # Assignment
            # -------------------------
            elif child.type in (
                "assignment",
                "augmented_assignment"
            ):
                entities.append({
                    "type": "assignment",
                    "start_line": child.start_point[0] + 1,
                    "end_line": child.end_point[0] + 1,
                    "text": self._text(child, source_code)
                })

            # -------------------------
            # Imports
            # -------------------------
            elif child.type in (
                "import_statement",
                "import_from_statement"
            ):
                entities.append({
                    "type": "import",
                    "start_line": child.start_point[0] + 1,
                    "end_line": child.end_point[0] + 1,
                    "text": self._text(child, source_code)
                })

        return entities

    # --------------------------------------------------
    # Class
    # --------------------------------------------------

    def _extract_class(self, node, source_code):

        name_node = node.child_by_field_name("name")

        body = node.child_by_field_name("body")

        return {
            "type": "class",
            "name": self._text(name_node, source_code)
                if name_node else None,

            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,

            "text": self._text(node, source_code),

            "children": self._extract_scope(
                body,
                source_code
            ) if body else []
        }

    # --------------------------------------------------
    # Function
    # --------------------------------------------------

    def _extract_function(self, node, source_code):

        name_node = node.child_by_field_name("name")

        body = node.child_by_field_name("body")

        return {
            "type": "function",
            "name": self._text(name_node, source_code)
                if name_node else None,

            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,

            "text": self._text(node, source_code),

            "children": self._extract_scope(
                body,
                source_code
            ) if body else []
        }

    # --------------------------------------------------
    # Decorated definition
    # --------------------------------------------------

    def _extract_decorated_definition(
        self,
        node,
        source_code
    ):

        decorators = []

        definition = None

        for child in node.children:

            if child.type == "decorator":
                decorators.append({
                    "type": "decorator",
                    "text": self._text(
                        child,
                        source_code
                    ),
                    "start_line": child.start_point[0] + 1,
                    "end_line": child.end_point[0] + 1
                })

            elif child.type in (
                "function_definition",
                "class_definition"
            ):
                definition = child

        if definition is None:
            return {
                "type": "decorated_definition",
                "text": self._text(node, source_code)
            }

        if definition.type == "class_definition":
            result = self._extract_class(
                definition,
                source_code
            )
        else:
            result = self._extract_function(
                definition,
                source_code
            )

        result["decorators"] = decorators

        # Include decorator in overall source range
        result["start_line"] = node.start_point[0] + 1

        return result

    # --------------------------------------------------
    # If / For / While
    # --------------------------------------------------

    def _extract_control_flow(
        self,
        node,
        source_code,
        control_type
    ):

        children = []

        for child in node.children:

            if child.type == "block":

                children.extend(
                    self._extract_scope(
                        child,
                        source_code
                    )
                )

            elif child.type == "elif_clause":

                children.append(
                    self._extract_control_flow(
                        child,
                        source_code,
                        "elif"
                    )
                )

            elif child.type == "else_clause":

                children.append(
                    self._extract_control_flow(
                        child,
                        source_code,
                        "else"
                    )
                )

        return {
            "type": control_type,

            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,

            "text": self._text(node, source_code),

            "children": children
        }

    # --------------------------------------------------
    # Utility
    # --------------------------------------------------

    @staticmethod
    def _text(node, source_code):

        if node is None:
            return None

        return source_code[
            node.start_byte:node.end_byte
        ].decode("utf-8", errors="replace")

In [2]:
file_path = '../test.py'

with open(file_path, 'rb') as f:
    source_code = f.read()

In [7]:
extractor = PythonExtractor()

extractor.extract(source_code=source_code)

{'type': 'file',
 'start_line': 1,
 'end_line': 583,
 'children': [{'type': 'comment',
   'start_line': 1,
   'end_line': 1,
   'text': '# ==========================================\r'},
  {'type': 'comment',
   'start_line': 2,
   'end_line': 2,
   'text': '# 1. BASIC VARIABLES\r'},
  {'type': 'comment',
   'start_line': 3,
   'end_line': 3,
   'text': '# ==========================================\r'},
  {'type': 'assignment',
   'start_line': 5,
   'end_line': 5,
   'text': 'name = "Rahul"'},
  {'type': 'assignment', 'start_line': 6, 'end_line': 6, 'text': 'age = 25'},
  {'type': 'assignment',
   'start_line': 7,
   'end_line': 7,
   'text': 'salary = 50000.0'},
  {'type': 'assignment',
   'start_line': 8,
   'end_line': 8,
   'text': 'is_employee = True'},
  {'type': 'assignment',
   'start_line': 10,
   'end_line': 10,
   'text': 'skills = ["Python", "SQL", "Git"]'},
  {'type': 'comment',
   'start_line': 18,
   'end_line': 18,
   'text': '# ========================================

In [2]:
from tree_sitter import Parser
from tree_sitter_language_pack import get_language


class PythonExtractor:
    """
    Extract meaningful Python code entities from a Tree-sitter syntax tree.

    The extractor intentionally does NOT expose every Tree-sitter AST node.
    It extracts structural/semantic entities that can later become graph
    nodes in Neo4j.
    """

    def __init__(self):
        self.language = get_language("python")
        self.parser = Parser(self.language)

    # ============================================================
    # PUBLIC API
    # ============================================================

    def extract(self, source_code: bytes):
        """
        Parse Python source code and return the extracted file scope.
        """

        tree = self.parser.parse(source_code)

        return {
            "type": "file",
            "children": self._extract_scope(tree.root_node, source_code),
        }

    # ============================================================
    # SCOPE EXTRACTION
    # ============================================================

    def _extract_scope(self, node, source_code):
        """
        Extract meaningful nodes from a scope.

        A scope can be:
        - module
        - class
        - function
        - if/else/elif body
        - loop body
        - try/except/finally body
        - with body
        - match/case body
        """

        children = []

        for child in node.children:

            # ----------------------------------------------------
            # Comments
            # ----------------------------------------------------
            if child.type == "comment":
                children.append({
                    "type": "comment",
                    "start_line": child.start_point[0] + 1,
                    "end_line": child.end_point[0] + 1,
                    "text": self._text(child, source_code),
                })

                continue

            # ----------------------------------------------------
            # Imports
            # ----------------------------------------------------
            if child.type in {
                "import_statement",
                "import_from_statement",
            }:
                children.append(
                    self._extract_import(child, source_code)
                )
                continue

            # ----------------------------------------------------
            # Assignments
            # ----------------------------------------------------
            if child.type in {
                "assignment",
                "augmented_assignment",
                "named_expression",
                "annotated_assignment",
            }:
                children.append(
                    self._extract_assignment(child, source_code)
                )
                continue

            # ----------------------------------------------------
            # Decorated definitions
            # ----------------------------------------------------
            if child.type == "decorated_definition":
                extracted = self._extract_decorated_definition(
                    child,
                    source_code
                )

                if extracted:
                    children.append(extracted)

                continue

            # ----------------------------------------------------
            # Normal class
            # ----------------------------------------------------
            if child.type == "class_definition":
                children.append(
                    self._extract_class(child, source_code)
                )
                continue

            # ----------------------------------------------------
            # Normal function
            # ----------------------------------------------------
            if child.type in {
                "function_definition",
                "async_function_definition",
            }:
                children.append(
                    self._extract_function(child, source_code)
                )
                continue

            # ----------------------------------------------------
            # IF / ELIF / ELSE
            # ----------------------------------------------------
            if child.type == "if_statement":
                children.append(
                    self._extract_if(child, source_code)
                )
                continue

            # ----------------------------------------------------
            # FOR
            # ----------------------------------------------------
            if child.type in {
                "for_statement",
                "async_for_statement",
            }:
                children.append(
                    self._extract_for(child, source_code)
                )
                continue

            # ----------------------------------------------------
            # WHILE
            # ----------------------------------------------------
            if child.type == "while_statement":
                children.append(
                    self._extract_while(child, source_code)
                )
                continue

            # ----------------------------------------------------
            # TRY / EXCEPT / FINALLY
            # ----------------------------------------------------
            if child.type == "try_statement":
                children.append(
                    self._extract_try(child, source_code)
                )
                continue

            # ----------------------------------------------------
            # WITH
            # ----------------------------------------------------
            if child.type in {
                "with_statement",
                "async_with_statement",
            }:
                children.append(
                    self._extract_with(child, source_code)
                )
                continue

            # ----------------------------------------------------
            # MATCH / CASE
            # ----------------------------------------------------
            if child.type == "match_statement":
                children.append(
                    self._extract_match(child, source_code)
                )
                continue

            # ----------------------------------------------------
            # Any other node
            #
            # We recurse because a meaningful construct can exist
            # inside another syntax wrapper.
            # ----------------------------------------------------
            nested = self._extract_scope(child, source_code)

            if nested:
                children.extend(nested)

        return children

    # ============================================================
    # IMPORTS
    # ============================================================

    def _extract_import(self, node, source_code):
        return {
            "type": "import",
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
        }

    # ============================================================
    # ASSIGNMENTS
    # ============================================================

    def _extract_assignment(self, node, source_code):
        result = {
            "type": "assignment",
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
        }

        return result

    # ============================================================
    # CLASS
    # ============================================================

    def _extract_class(self, node, source_code):
        name = None

        for child in node.children:
            if child.type == "identifier":
                name = self._text(child, source_code)
                break

        result = {
            "type": "class",
            "name": name,
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
            "children": [],
        }

        # --------------------------------------------------------
        # Extract class body
        # --------------------------------------------------------

        body = self._find_child(node, "block")

        if body:
            result["children"] = self._extract_scope(
                body,
                source_code
            )

        # --------------------------------------------------------
        # Extract inheritance
        # --------------------------------------------------------

        superclasses = []

        for child in node.children:
            if child.type == "argument_list":
                for argument in child.children:
                    if argument.type not in {
                        "(",
                        ")",
                        ",",
                    }:
                        superclasses.append(
                            self._text(argument, source_code)
                        )

        if superclasses:
            result["bases"] = superclasses

        return result

    # ============================================================
    # FUNCTION
    # ============================================================

    def _extract_function(self, node, source_code):
        name = None

        for child in node.children:

            if child.type == "identifier":
                name = self._text(child, source_code)
                break

        result = {
            "type": "function",
            "name": name,
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
            "children": [],
        }

        # --------------------------------------------------------
        # Async function
        # --------------------------------------------------------

        if node.type == "async_function_definition":
            result["async"] = True

        # --------------------------------------------------------
        # Extract parameters
        # --------------------------------------------------------

        parameters = self._find_child(node, "parameters")

        if parameters:
            result["parameters"] = self._text(
                parameters,
                source_code
            )

        # --------------------------------------------------------
        # Extract return type
        # --------------------------------------------------------

        return_type = self._find_child(
            node,
            "type"
        )

        if return_type:
            result["return_type"] = self._text(
                return_type,
                source_code
            )

        # --------------------------------------------------------
        # Extract function body
        # --------------------------------------------------------

        body = self._find_child(node, "block")

        if body:
            result["children"] = self._extract_scope(
                body,
                source_code
            )

        return result

    # ============================================================
    # DECORATED DEFINITION
    # ============================================================

    def _extract_decorated_definition(
        self,
        node,
        source_code
    ):
        """
        Handles:

            @decorator
            def func():
                ...

        and:

            @decorator(...)
            class MyClass:
                ...
        """

        decorators = []

        definition = None

        for child in node.children:

            if child.type == "decorator":
                decorators.append({
                    "type": "decorator",
                    "text": self._text(
                        child,
                        source_code
                    ),
                    "start_line": child.start_point[0] + 1,
                    "end_line": child.end_point[0] + 1,
                })

            elif child.type in {
                "function_definition",
                "async_function_definition",
                "class_definition",
            }:
                definition = child

        if definition is None:
            return None

        # --------------------------------------------------------
        # Extract actual definition
        # --------------------------------------------------------

        if definition.type in {
            "function_definition",
            "async_function_definition",
        }:
            result = self._extract_function(
                definition,
                source_code
            )

        else:
            result = self._extract_class(
                definition,
                source_code
            )

        # --------------------------------------------------------
        # Attach decorators
        # --------------------------------------------------------

        if decorators:
            result["decorators"] = decorators

        # The decorated definition starts at the first decorator.
        result["start_line"] = node.start_point[0] + 1
        result["end_line"] = node.end_point[0] + 1
        result["text"] = self._text(node, source_code)

        return result

    # ============================================================
    # IF / ELIF / ELSE
    # ============================================================

    def _extract_if(self, node, source_code):
        result = {
            "type": "if",
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
            "children": [],
        }

        for child in node.children:

            # Main IF body
            if child.type == "block":
                result["children"].extend(
                    self._extract_scope(
                        child,
                        source_code
                    )
                )

            # elif
            elif child.type == "elif_clause":
                result["children"].append(
                    self._extract_elif(
                        child,
                        source_code
                    )
                )

            # else
            elif child.type == "else_clause":
                result["children"].append(
                    self._extract_else(
                        child,
                        source_code
                    )
                )

        return result

    def _extract_elif(self, node, source_code):
        result = {
            "type": "elif",
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
            "children": [],
        }

        body = self._find_child(node, "block")

        if body:
            result["children"] = self._extract_scope(
                body,
                source_code
            )

        return result

    def _extract_else(self, node, source_code):
        result = {
            "type": "else",
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
            "children": [],
        }

        body = self._find_child(node, "block")

        if body:
            result["children"] = self._extract_scope(
                body,
                source_code
            )

        return result

    # ============================================================
    # FOR
    # ============================================================

    def _extract_for(self, node, source_code):
        result = {
            "type": "for",
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
            "children": [],
        }

        if node.type == "async_for_statement":
            result["async"] = True

        body = self._find_child(node, "block")

        if body:
            result["children"] = self._extract_scope(
                body,
                source_code
            )

        return result

    # ============================================================
    # WHILE
    # ============================================================

    def _extract_while(self, node, source_code):
        result = {
            "type": "while",
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
            "children": [],
        }

        body = self._find_child(node, "block")

        if body:
            result["children"] = self._extract_scope(
                body,
                source_code
            )

        return result

    # ============================================================
    # TRY / EXCEPT / ELSE / FINALLY
    # ============================================================

    def _extract_try(self, node, source_code):
        """
        Handles:

            try:
                ...

            except ValueError:
                ...

            except Exception as e:
                ...

            else:
                ...

            finally:
                ...

        Also handles except* clauses using the same generic
        exception-clause extraction.
        """

        result = {
            "type": "try",
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
            "children": [],
        }

        for child in node.children:

            # Main try body
            if child.type == "block":
                result["children"].extend(
                    self._extract_scope(
                        child,
                        source_code
                    )
                )

            # except / except*
            elif child.type in {
                "except_clause",
                "except_group_clause",
            }:
                result["children"].append(
                    self._extract_except(
                        child,
                        source_code
                    )
                )

            # else attached to try
            elif child.type == "else_clause":
                result["children"].append(
                    self._extract_try_else(
                        child,
                        source_code
                    )
                )

            # finally
            elif child.type == "finally_clause":
                result["children"].append(
                    self._extract_finally(
                        child,
                        source_code
                    )
                )

        return result

    def _extract_except(self, node, source_code):
        result = {
            "type": "except",
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
            "children": [],
        }

        # --------------------------------------------------------
        # Extract exception type / name where available
        # --------------------------------------------------------

        exception_types = []

        for child in node.children:

            if child.type in {
                "identifier",
                "attribute",
                "tuple",
                "list",
                "as_pattern",
            }:
                exception_types.append(
                    self._text(child, source_code)
                )

        if exception_types:
            result["exception"] = exception_types

        # --------------------------------------------------------
        # Extract except body
        # --------------------------------------------------------

        body = self._find_child(node, "block")

        if body:
            result["children"] = self._extract_scope(
                body,
                source_code
            )

        return result

    def _extract_try_else(self, node, source_code):
        result = {
            "type": "else",
            "context": "try",
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
            "children": [],
        }

        body = self._find_child(node, "block")

        if body:
            result["children"] = self._extract_scope(
                body,
                source_code
            )

        return result

    def _extract_finally(self, node, source_code):
        result = {
            "type": "finally",
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
            "children": [],
        }

        body = self._find_child(node, "block")

        if body:
            result["children"] = self._extract_scope(
                body,
                source_code
            )

        return result

    # ============================================================
    # WITH
    # ============================================================

    def _extract_with(self, node, source_code):
        """
        Handles:

            with open("file.txt") as f:
                ...

        and:

            with A() as a, B() as b:
                ...

        and async with.
        """

        result = {
            "type": "with",
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
            "children": [],
        }

        if node.type == "async_with_statement":
            result["async"] = True

        # --------------------------------------------------------
        # Extract with clauses
        # --------------------------------------------------------

        items = []

        for child in node.children:

            if child.type in {
                "with_clause",
                "with_item",
            }:
                items.append({
                    "type": "with_item",
                    "text": self._text(
                        child,
                        source_code
                    ),
                    "start_line": child.start_point[0] + 1,
                    "end_line": child.end_point[0] + 1,
                })

        if items:
            result["items"] = items

        # --------------------------------------------------------
        # Extract body
        # --------------------------------------------------------

        body = self._find_child(node, "block")

        if body:
            result["children"] = self._extract_scope(
                body,
                source_code
            )

        return result

    # ============================================================
    # MATCH / CASE
    # ============================================================

    def _extract_match(self, node, source_code):
        """
        Handles Python 3.10+:

            match value:
                case 1:
                    ...
                case _:
                    ...
        """

        result = {
            "type": "match",
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
            "children": [],
        }

        for child in node.children:

            if child.type == "case_clause":
                result["children"].append(
                    self._extract_case(
                        child,
                        source_code
                    )
                )

            elif child.type == "block":
                result["children"].extend(
                    self._extract_scope(
                        child,
                        source_code
                    )
                )

        return result

    def _extract_case(self, node, source_code):
        result = {
            "type": "case",
            "start_line": node.start_point[0] + 1,
            "end_line": node.end_point[0] + 1,
            "text": self._text(node, source_code),
            "children": [],
        }

        body = self._find_child(node, "block")

        if body:
            result["children"] = self._extract_scope(
                body,
                source_code
            )

        return result

    # ============================================================
    # HELPERS
    # ============================================================

    def _find_child(self, node, child_type):
        """
        Find the first direct child with the requested Tree-sitter
        node type.
        """

        for child in node.children:
            if child.type == child_type:
                return child

        return None

    def _text(self, node, source_code):
        """
        Return the exact source text represented by a Tree-sitter node.
        """

        return source_code[
            node.start_byte:node.end_byte
        ].decode(
            "utf-8",
            errors="replace"
        )

In [4]:
file_path = '../test_codes/test.py'

with open(file_path, 'rb') as f:
    source_code = f.read()

In [5]:
extractor = PythonExtractor()

extractor.extract(source_code=source_code)

{'type': 'file',
 'children': [{'type': 'comment',
   'start_line': 1,
   'end_line': 1,
   'text': '# ==========================================\r'},
  {'type': 'comment',
   'start_line': 2,
   'end_line': 2,
   'text': '# 1. BASIC VARIABLES\r'},
  {'type': 'comment',
   'start_line': 3,
   'end_line': 3,
   'text': '# ==========================================\r'},
  {'type': 'assignment',
   'start_line': 5,
   'end_line': 5,
   'text': 'name = "Rahul"'},
  {'type': 'assignment', 'start_line': 6, 'end_line': 6, 'text': 'age = 25'},
  {'type': 'assignment',
   'start_line': 7,
   'end_line': 7,
   'text': 'salary = 50000.0'},
  {'type': 'assignment',
   'start_line': 8,
   'end_line': 8,
   'text': 'is_employee = True'},
  {'type': 'assignment',
   'start_line': 10,
   'end_line': 10,
   'text': 'skills = ["Python", "SQL", "Git"]'},
  {'type': 'comment',
   'start_line': 18,
   'end_line': 18,
   'text': '# ==========================================\r'},
  {'type': 'comment',
   'st